In [1]:
MODULE_DIR = '/Users/ravitpichayavet/Documents/GaTechOR/GraduateResearch/CTC_CVRP/Modules'
GUROBI_LICENSE_DIR = '/Users/ravitpichayavet/gurobi.lic'

import sys
sys.path.insert(0,MODULE_DIR)
import importlib

import visualize_sol as vis_sol
import initialize_path as init_path
import random_instance as rand_inst
import utility as util
import branch_and_price as bnp
import model as md

import numpy as np
from gurobipy import *
import os
os.environ['GRB_LICENSE_FILE'] = GUROBI_LICENSE_DIR


from itertools import combinations,permutations 
import random 
import pandas as pd
import itertools
import plotly.graph_objects as go
import networkx as nx
import plotly.offline as py 
import pickle as pk
import nltk
import time
import copy

from matplotlib import pyplot as plt
from sklearn.datasets.samples_generator import make_blobs
from sklearn.cluster import KMeans
from scipy.spatial import distance
import logging
# logging.basicConfig(filename='myapp.log',filemode='w',format='%(message)s',level=logging.INFO)
# print = logging.info

/Users/ravitpichayavet/opt/miniconda3/lib/python3.8/site-packages/sklearn/utils/deprecation.py:143: FutureWarning:

The sklearn.datasets.samples_generator module is  deprecated in version 0.22 and will be removed in version 0.24. The corresponding classes / functions should instead be imported from sklearn.datasets. Anything that cannot be imported from sklearn.datasets is now part of the private API.



## (0) Parameters and Constants

- Service region 20x20 [mile sq.]
- #Demand nodes, $n = 10$ (uniformly distributed)
- Manhattan distance ($l1$)
- Vehicle speed, $v = 20$ [mile/hr], $c_{(i,j)} = \frac{1}{20}(|x_{i}-x_{j}|+|y_{i}-y_{j}|)$ [hr]
- $s_{0}=0.5$ [hr], $s_{1} = 0$ 
- $q_{i} \in \{1,2,3,4,5\}$ [packges/hr]
- Maximum carrying capacity, $Q = 20$ [packages]


In [2]:
no_dock = 0 
no_layer = 1
no_demand_node = 6
truck_capacity = 20
fixed_setup_time = 0.5
truck_speed = 20

constant_dict = { 'truck_capacity' : truck_capacity,'fixed_setup_time' : fixed_setup_time,
                    'truck_speed': truck_speed,'max_vehicles':7}

edge_plot_config = {'line_width':2.5, 'line_color':'#a26337', 'dash':None, 'name':''}

# Create network components: nodes, arcs
labeling_dict = vis_sol.create_nodes(no_dock,no_demand_node)
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()

In [39]:
## SKIP THIS IF DON'T WANT TO USE NEW RANDOM DISTANCE MAP

# Random distance matrix
distance_matrix, node_position = rand_inst.rand_uniform_dis_mat(nodes,depot[0])
# Random customer demand
_depot_demand = pd.Series([0,0,0], index =all_depot+depot)
customer_demand = rand_inst.rand_cust_demand(customers).append(_depot_demand)
# Visualization 
color_map = vis_sol.create_color_list(nodes,{'O':4, 'c':2})
node_trace = vis_sol.create_node_trace(node_position,color_map,_symbol_dict={'O':'diamond', 'c':'circle'})
print(color_map, node_trace)

{'c_2': 2, 'c_5': 2, 'c_4': 2, 'c_3': 2, 'c_6': 2, 'c_1': 2, 'O': 4} Scatter({
    'marker': {'color': [2, 2, 2, 2, 2, 2, 4],
               'colorscale': [[0.0, 'rgb(0,0,0)'], [0.25, 'rgb(230,0,0)'], [0.5,
                              'rgb(230,210,0)'], [0.75, 'rgb(255,255,255)'], [1.0,
                              'rgb(160,200,255)']],
               'line': {'width': 1},
               'reversescale': True,
               'showscale': False,
               'size': 30,
               'symbol': [circle, circle, circle, circle, circle, circle, diamond]},
    'mode': 'markers+text',
    'name': 'Node',
    'text': [c_2, c_5, c_4, c_3, c_6, c_1, O],
    'textposition': 'top center',
    'x': array([19.97616043, 12.12042968, 12.87885641,  5.54721174, 19.94284358,
                 3.21669417, 10.        ]),
    'y': array([14.75912294,  4.95810549, 15.11374658, 16.53774029, 11.1528769 ,
                14.12923078, 10.        ])
})


### Save Instance to pickle

In [41]:
instance_name='DemoInstanceCus{0}_1'.format(no_demand_node);print(instance_name);
instance_data = {'distance_matrix':distance_matrix,
               'node_position':node_position,
               'node_trace':node_trace,
                'customer_demand_df':customer_demand}
# instance_data

DemoInstanceCus6_1


In [42]:
with open('./Instances/%s.pickle'%instance_name,'wb') as f1:
    pk.dump(instance_data,f1)
    print(instance_name+" is saved succuessfully!")

DemoInstanceCus6_1 is saved succuessfully!


### Import instance

In [43]:
instance_name='DemoInstanceCus{0}_1'.format(no_demand_node);print(instance_name);
with open('./Instances/%s.pickle'%instance_name,'rb') as f1:
    r_instance = pk.load(f1)

distance_matrix = r_instance['distance_matrix']
node_trace = r_instance['node_trace']
customer_demand = r_instance['customer_demand_df']

DemoInstanceCus6_1


### Plot network

In [60]:
m1 = sys.modules['visualize_sol']
m2 = sys.modules['random_instance']
m3 = sys.modules['initialize_path']
m4 = sys.modules['model']
importlib.reload(m1)
importlib.reload(m2)
importlib.reload(m3)
importlib.reload(m4)

<module 'model' from '/Users/ravitpichayavet/Documents/GaTechOR/GraduateResearch/CTC_CVRP/Modules/model.py'>

In [40]:
path1 = {'arcs_list': [('c_4','c_2'),('c_2','O')],
                      'config': {'line_width':2.5, 'line_color':'#a26337', 'dash':None, 'name':''}}
path2 = {'arcs_list': [('O','c_6'),('c_6','c_3'),('c_3','O')],
                      'config': {'line_width':2.5, 'line_color':'#a26337', 'dash':None, 'name':''}}
path_arcs_list = [path1,path2]
vis_sol.plot_network(path_arcs_list, node_trace,_cus_dem=customer_demand,_title=instance_name)

## (1) Initial solution set $\mathbb{R}^{+}$

In [44]:
initializer = init_path.InitialRouteGenerator(no_layer,labeling_dict,
                                              customer_demand,constant_dict,
                                              distance_matrix)

In [45]:
# Generate initial set of feasilble routes
row_labels = ['lr','m']+depot+customers+arcs
x = pd.DataFrame(data =row_labels,columns=['labels'])
initializer.generateRoutes(initRouteDf=x,
                           truck_cap_limit=truck_capacity,
                           max_visited_nodes=6, max_vehicles_per_route=6,mode='all')
x = x.set_index('labels')
# Reformatting
feasibleCols = x.shape[1]
init_route = initializer.generateBasicInitialPatterns(feasibleCols,initRouteDf=x)
init_route.rename(index=lambda x:'route[%d]'%x,inplace=True)

nbInitRoute is set to (#UniqueSequences) * (#MaxVehiclesPerRoute) = 1956*6 = 11736
progress: 0 / 11736
progress: 5868 / 11736
#Feasible Cols: 6746
Elapsed-time: 9.166460037231445


In [52]:
x.head(10)

,route[0],route[1],route[2],route[3],route[4],route[5],route[6],route[7],route[8],route[9],...,route[6736],route[6737],route[6738],route[6739],route[6740],route[6741],route[6742],route[6743],route[6744],route[6745]
labels,,,,,,,,,,,,,,,,,,,,,
lr,1.591254,1.591254,1.591254,1.591254,1.591254,1.591254,1.973528,1.973528,1.973528,1.973528,...,4.329891,4.329891,4.266902,4.266902,4.266902,4.582314,4.582314,4.582314,4.964604,4.964604
m,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000,1.000000,2.000000,3.000000,4.000000,...,5.000000,6.000000,4.000000,5.000000,6.000000,4.000000,5.000000,6.000000,5.000000,6.000000
O,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
c_6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Plot sample of route generated

In [53]:
sample_r = x['route[1000]']
sample_arcs = sample_r[sample_r.index.isin(arcs)][sample_r==1]

route_plot_dict = dict(zip(['arcs_list','config'],
                           [sample_arcs.index.to_list(),edge_plot_config]))
# {arcs_list: [('O','c_1'),('c_1','c_2'),('c_2','O')]
#                           config: {'line_width':.., 'line_color':..., 'dash':..., 'name':...}}
# reformatted_arcs = [merge_depot_arcs_var([t],depot,depot_s,depot_t) for t in sample_arcs.index.to_list()]
print('mr:',sample_r['m'], 'lr:',sample_r['lr'])
vis_sol.plot_network([route_plot_dict],node_trace,_title="Demo Instance",_cus_dem=customer_demand)

mr: 3.0 lr: 3.8098223999819667


## (2) Find $m_{max}$ using only TRP Objective

2.1) $m_{max} = 6$

In [68]:
constant_dict['max_vehicles'] = 6
print(constant_dict)

{'truck_capacity': 20, 'fixed_setup_time': 0.5, 'truck_speed': 20, 'max_vehicles': 6}


In [65]:
TRP_model2_1 = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict,_mode = 'TRPOnly')
TRP_model2_1.buildModel()
timeLimit=120;GAP=0.01;TRP_model2_1.solveModel()

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 8.959028005599976
Finish generating objective!
Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 7 rows, 6746 columns and 38636 nonzeros
Model fingerprint: 0xf4044048
Variable types: 0 continuous, 6746 integer (6746 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [6e-01, 6e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]
Found heuristic solution: objective 41.4346048
Presolve removed 0 rows and 525 columns
Presolve time: 0.19s
Presolved: 7 rows, 6221 columns, 35593 nonzeros
Variable types: 0 continuous, 6221 integer (6221 binary)
Found heuristic solution: objective 8.3826468

Root relaxation: objective 8.382647e+00, 7 iterations, 0.00 

In [66]:
op_routes = getOptimalSolution(TRP_model2_1)
plotSolution(TRP_model2_1,op_routes,constant_dict, _graph_name="Demo Instance")

        route[1]  route[12]  route[18]  route[24]  route[138]
labels                                                       
m       2.000000   1.000000    1.00000   1.000000    1.000000
lr      1.591254   1.599053    1.29926   1.216232    1.973528


2.2) $m_{max} = 7$

In [57]:
constant_dict['max_vehicles'] = 10
print(constant_dict)

{'truck_capacity': 20, 'fixed_setup_time': 0.5, 'truck_speed': 20, 'max_vehicles': 10}


In [58]:
TRP_model2_2 = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict,_mode = 'TRPOnly')
TRP_model2_2.buildModel()
timeLimit=120;GAP=0.01;TRP_model2_2.solveModel()

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 9.142796993255615
Finish generating objective!
Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 7 rows, 6746 columns and 38636 nonzeros
Model fingerprint: 0x63050e80
Variable types: 0 continuous, 6746 integer (6746 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [6e-01, 6e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+01]
Found heuristic solution: objective 30.2210555
Presolve time: 0.03s
Presolved: 7 rows, 6746 columns, 38623 nonzeros
Variable types: 0 continuous, 6746 integer (6746 binary)
Found heuristic solution: objective 8.3826468

Root relaxation: objective 8.382647e+00, 6 iterations, 0.00 seconds

    Nodes    |    Current Node 

In [59]:
op_routes = getOptimalSolution(TRP_model2_2)
plotSolution(TRP_model2_2,getOptimalSolution(TRP_model2_2),constant_dict, _graph_name="Demo Instance")

        route[0]  route[6]  route[12]  route[22]  route[24]  route[30]
labels                                                                
m       1.000000  1.000000   1.000000    5.00000   1.000000   1.000000
lr      1.591254  1.973528   1.599053    1.29926   1.216232   1.609572


In [24]:
TRP_model2_1.init_routes_df.set_index('labels')['route[110]'].head(9), TRP_model2_1.route_cost[110]

(labels
 lr     1.960708
 m      1.000000
 O      1.000000
 c_1    0.000000
 c_2    1.000000
 c_3    0.000000
 c_4    1.000000
 c_5    0.000000
 c_6    0.000000
 Name: route[110], dtype: float64,
 {'total_cost': 9.719138637606381,
  'avg_waiting': 5.882124378386067,
  'avg_travel': 3.837014259220315})

In [38]:
TRP_model2_1.route_cost[10]['avg_travel']+TRP_model2_1.route_cost[18]['avg_travel']

3.837014259220316

In [36]:
TRP_model2_1.init_routes_df.set_index('labels').loc[:,['route[10]','route[18]']].head(9), TRP_model2_1.route_cost[10],TRP_model2_1.route_cost[18]

(        route[10]  route[18]
 labels                      
 lr       1.960708   1.415598
 m        5.000000   1.000000
 O        1.000000   1.000000
 c_1      0.000000   0.000000
 c_2      1.000000   0.000000
 c_3      0.000000   0.000000
 c_4      0.000000   1.000000
 c_5      0.000000   0.000000
 c_6      0.000000   0.000000,
 {'total_cost': 3.705699502708853,
  'avg_waiting': 0.7842832504514755,
  'avg_travel': 2.9214162522573774},
 {'total_cost': 2.3311960139258767,
  'avg_waiting': 1.4155980069629384,
  'avg_travel': 0.9155980069629382})

In [37]:
customer_demand

c_1    1
c_2    4
c_3    3
c_4    2
c_5    5
c_6    5
O_s    0
O_t    0
O      0
dtype: int64

In [12]:
def getOptimalSolution(_model):
    _sol_obj = pd.Series(_model.model.getVars())
    _sol_vec = pd.DataFrame(index = _sol_obj.apply(lambda x:x.VarName))
    _sol_vec['value'] = _sol_obj.apply(lambda x:x.X).values
    optimal_routes = _sol_vec.loc[_sol_vec['value']==1]
    return optimal_routes

def plotSolution(_model,_optimal_routes,_constant_dict, _graph_name="Demo Instance"):
    print(_model.init_routes_df.set_index('labels').loc[['m','lr']][_optimal_routes.index])
    _formatted_routes_list =  _model.getRoute4Plot(_optimal_routes.index.to_list(),
                                                        _model.init_routes_df.set_index('labels'),edge_plot_config)
    vis_sol.plot_network(_formatted_routes_list,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

In [113]:
sol_obj_phaseII = pd.Series(TRP_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
optimal_routes_phaseII.index

Index(['route[0]', 'route[12]', 'route[24]', 'route[30]', 'route[110]'], dtype='object')

In [114]:
print(x.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  TRP_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        x,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

        route[0]  route[12]  route[24]  route[30]  route[110]
labels                                                       
m       1.000000   1.000000   1.000000   1.000000    1.000000
lr      2.275679   1.486525   1.739945   1.047411    1.960708


In [97]:
TRP_model.init_routes_df.set_index('labels')['route[10]'].head(9), TRP_model.route_cost[10]

(labels
 lr     1.960708
 m      5.000000
 O      1.000000
 c_1    0.000000
 c_2    1.000000
 c_3    0.000000
 c_4    0.000000
 c_5    0.000000
 c_6    0.000000
 Name: route[10], dtype: float64,
 {'total_cost': 3.705699502708853,
  'avg_waiting': 0.7842832504514755,
  'avg_travel': 2.9214162522573774})

In [115]:
TRP_model.route_cost[]

{'total_cost': 3.709574703986009,
 'avg_waiting': 2.2297873519930045,
 'avg_travel': 1.4797873519930045}

In [102]:
4*distance_matrix[('c_2','O')]/constant_dict['truck_speed']

2.9214162522573774

In [106]:
(2*distance_matrix[('c_2','O')]/constant_dict['truck_speed'])+constant_dict['fixed_setup_time']

1.9607081261286887

In [107]:
customer_demand['c_2']*(constant_dict['fixed_setup_time']+(2*distance_matrix[('c_2','O')]/constant_dict['truck_speed']))/constant_dict['truck_capacity']

0.39214162522573776

In [88]:
customer_demand

c_1    1
c_2    4
c_3    3
c_4    2
c_5    5
c_6    5
O_s    0
O_t    0
O      0
dtype: int64

In [76]:
print(x.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  TRP_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        x,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

        route[0]  route[3864]
labels                       
m       1.000000     5.000000
lr      2.275679     3.096154


In [15]:
phaseI_model = md.phaseIModel(init_route, initializer,
                 distance_matrix,constant_dict)

Academic license - for non-commercial use only - expires 2022-08-13
Using license file /Users/ravitpichayavet/gurobi.lic


### (2.1) Build-up model

In [16]:
phaseI_model.buildModel()

Finish generating variables!
Finish generating constrains!
Finish generating objective!


### (2.2) Solve model

In [17]:
phaseI_model.solveModel(np.inf,0.1)

Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Parameter TImeLimit unchanged
   Value: inf  Min: 0.0  Max: inf  Default: inf
Changed value of parameter MIPGap to 0.1
   Prev: 0.0001  Min: 0.0  Max: inf  Default: 0.0001
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 2 rows, 8 columns and 12 nonzeros
Model fingerprint: 0x49a52e71
Variable types: 0 continuous, 8 integer (8 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 2.0000000
Presolve time: 0.00s
Presolved: 2 rows, 8 columns, 12 nonzeros
Variable types: 0 continuous, 8 integer (8 binary)

Root relaxation: objective 1.000000e+00, 2 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexp

In [18]:
sol_obj_phaseI = pd.Series(phaseI_model.model.getVars())
sol_vec_phaseI = pd.DataFrame(index = sol_obj_phaseI.apply(lambda x:x.VarName))
sol_vec_phaseI['value'] = sol_obj_phaseI.apply(lambda x:x.X).values
optimal_routes_phaseI = sol_vec_phaseI.loc[sol_vec_phaseI['value']==1]
optimal_routes_phaseI.index

Index(['route[4]'], dtype='object')

In [19]:
print(x.loc[['m','lr']][optimal_routes_phaseI.index])
formatted_routes_list_I = phaseI_model.getRoute4Plot(_route_name_list=optimal_routes_phaseI.index.to_list(), _colums_df=x,_route_config=edge_plot_config)
vis_sol.plot_network(formatted_routes_list_I,node_trace,_title="Demo Instance",_cus_dem=customer_demand)


        route[4]
labels          
m       1.000000
lr      2.098735


## (3) Phase II: Cycle time VRP
(Set partitioning)

In [283]:
phaseII_model = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict)

### (3.1) Build-up model

In [284]:
phaseII_model.buildModel()

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 0.15954113006591797
Finish generating objective!


### (3.2) Solve model

In [285]:
timeLimit=120;GAP=0.01;phaseII_model.solveModel()

Changed value of parameter PoolSearchMode to 2
   Prev: 0  Min: 0  Max: 2  Default: 0
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 6 rows, 150 columns and 420 nonzeros
Model fingerprint: 0xb1a62e92
Variable types: 0 continuous, 150 integer (150 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [4e-01, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]
Found heuristic solution: objective 18.0365188
Presolve removed 0 rows and 25 columns
Presolve time: 0.01s
Presolved: 6 rows, 125 columns, 341 nonzeros
Variable types: 0 continuous, 125 integer (125 binary)
Found heuristic solution: objective 16.2039465
Found heuristic solution: objective 14.2481907

Root relaxation: objective 1.167592e+01, 16 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf 

In [286]:
sol_obj_phaseII = pd.Series(phaseII_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
optimal_routes_phaseII.index

Index(['route[1]', 'route[12]', 'route[19]', 'route[108]'], dtype='object')

In [287]:
print(x.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  phaseII_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        x,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

        route[1]  route[12]  route[19]  route[108]
labels                                            
m       2.000000   1.000000   2.000000     1.00000
lr      1.779958   1.296562   2.017318     1.63051


In [78]:
0.5+(distance_matrix[('O','c_2')]*2+distance_matrix[('c_2','c_1')])/20

2.168291731499769

In [268]:
# x

In [45]:
# phaseII_model.init_routes_df.set_index('labels')['route[54]']

In [269]:
total_cost_phaseI_sol = phaseII_model.route_cost[optimal_routes_phaseI.index].sum()
total_cost_phaseII_sol = phaseII_model.route_cost[optimal_routes_phaseII.index].sum()
print('cost of phase I sol:',total_cost_phaseI_sol,
                              ", Avg mins:",total_cost_phaseI_sol*60/customer_demand.sum(),
     '\ncost of phase II sol:', total_cost_phaseII_sol,
                                  ", Avg mins:",total_cost_phaseII_sol*60/customer_demand.sum())

cost of phase I sol: 1.4707487370187484 , Avg mins: 11.030615527640613 
cost of phase II sol: 7.198262271731572 , Avg mins: 53.98696703798679


In [26]:
# [(('O',x),distance_matrix[('O',x)]) for x in customers], customer_demand
# a = np.sum([(customer_demand['c_'+str(i)]*(0.5+(distance_matrix[('O','c_'+str(i))]/10)))/6 for i in range(1,6)])
# b = np.sum([(customer_demand['c_'+str(i)]*((distance_matrix[('O','c_'+str(i))]/20))) for i in range(1,6)])
# a+b

In [13]:
m1 = sys.modules['visualize_sol']
m2 = sys.modules['random_instance']
m3 = sys.modules['initialize_path']
m4 = sys.modules['model']
importlib.reload(m1)
importlib.reload(m2)
importlib.reload(m3)
importlib.reload(m4)

<module 'model' from '/Users/ravitpichayavet/Documents/GaTechOR/GraduateResearch/CTC_CVRP/Modules/model.py'>